### Link github: https://github.com/henrynlh/Artificial-Intelligence.git

### Import thư viện

In [1]:
import random

### Random trạng thái ban đầu

In [ ]:
def input_state():
    m = int(input("Nhập số dòng (m): "))
    n = int(input("Nhập số cột (n): "))

    obstacle_rate = 0.2  # 20% là vật cản x
    dirty_rate = 0.4     # 40% là ô bẩn 1

    floor = []

    for i in range(m):
        row = []
        for j in range(n):
            r = random.random()

            if r < obstacle_rate:
                row.append("x")      # vật cản
            elif r < obstacle_rate + dirty_rate:
                row.append(1)        # ô bẩn
            else:
                row.append(0)        # ô sạch

        floor.append(row)

    valid_positions = []
    #Tìm danh sách vị trí hợp lệ máy hút bụi có thể đứng
    for i in range(m):
        for j in range(n):
            if floor[i][j] != "x":
                valid_positions.append((i, j))

    if not valid_positions:
        print("Ma trận toàn vật cản, máy hút bụi không thể bắt đầu.")
        return None

    pos = random.choice(valid_positions)

    state = {
        "floor": floor, #Lưu ma trận sàn nhà
        "pos": pos,     #Lưu vị trí máy hút bụi
        "m": m,         #Lưu số dòng
        "n": n          #Lưu số cột
    }

    return state

### Các hàm xử lý trạng thái

In [3]:
def copy_state(state):
    return {
        "floor": [row[:] for row in state["floor"]],
        "pos": state["pos"],
        "m": state["m"],
        "n": state["n"]
    }

In [4]:
def print_state(state):
    floor = state["floor"]
    pos = state["pos"]

    print("0 = sạch, 1 = bẩn, x = vật cản, R = máy hút bụi")

    for i in range(state["m"]):
        row = []
        for j in range(state["n"]):
            if (i, j) == pos:
                row.append("R")
            else:
                row.append(floor[i][j])
        print(row)

    print("Vị trí máy hút bụi:", pos)

In [ ]:
#Hàm chuyển state thành key để lưu vào visited
def state_key(state):
    return (
        state["pos"],
        tuple(tuple(row) for row in state["floor"])
    )

In [6]:
def is_clean_state(state):
    for row in state["floor"]:
        for cell in row:
            if cell == 1:
                return False
    return True

### Model

In [7]:
def get_possible_moves(state):
    x, y = state["pos"]
    floor = state["floor"]
    m = state["m"]
    n = state["n"]

    possible_moves = []

    if x > 0 and floor[x - 1][y] != "x":
        possible_moves.append("UP")

    if x < m - 1 and floor[x + 1][y] != "x":
        possible_moves.append("DOWN")

    if y > 0 and floor[x][y - 1] != "x":
        possible_moves.append("LEFT")

    if y < n - 1 and floor[x][y + 1] != "x":
        possible_moves.append("RIGHT")

    return possible_moves

In [8]:
def apply_move(pos, action):
    x, y = pos

    if action == "UP":
        x -= 1
    elif action == "DOWN":
        x += 1
    elif action == "LEFT":
        y -= 1
    elif action == "RIGHT":
        y += 1

    return (x, y)

In [9]:
def model(state, action):
    """
    model: mô hình dự đoán trạng thái mới
    input: trạng thái hiện tại + hành động
    output: trạng thái mới sau khi hành động
    """

    if action is None:
        return state

    new_state = copy_state(state)

    new_pos = apply_move(new_state["pos"], action)
    new_state["pos"] = new_pos

    x, y = new_pos

    # Nếu đi tới ô bẩn thì hút bụi luôn
    if new_state["floor"][x][y] == 1:
        new_state["floor"][x][y] = 0

    return new_state

## Model-Based Reflex Agent

### Update-state

In [19]:
def update_state(previous_state, previous_action, percept, model):
    # Vì agent quan sát được toàn bộ trạng thái hiện tại
    # nên percept chính là state mới
    return percept

### Rule-match

In [17]:
# Hàm chọn luật phù hợp
def rule_match(state, rules, visited):

    """
    Luật:
    1. Nếu có hướng đi tới ô bẩn và trạng thái đó chưa đi -> ưu tiên chọn.
    2. Nếu không có ô bẩn xung quanh -> chọn random hướng chưa đi.
    3. Nếu tất cả hướng đều lặp -> dừng.
    """

    # Lấy các hướng có thể đi từ trạng thái hiện tại
    possible_moves = get_possible_moves(state)

    # In ra các bước có thể đi
    print("\nCác bước có thể đi:", possible_moves)

    # Nếu không có hướng nào đi được
    if not possible_moves:

        # Trả về action None để báo dừng
        return {
            "condition": "Không còn bước đi hợp lệ",
            "action": None
        }

    # Danh sách lưu hướng đi tới ô bẩn
    dirty_moves = []

    # Danh sách lưu hướng chưa từng đi
    unvisited_moves = []

    # Danh sách lưu hướng bị lặp lại
    duplicated_moves = []

    # Duyệt từng hướng có thể đi
    for action in possible_moves:

        # Tính vị trí mới nếu đi theo hướng đó
        new_pos = apply_move(state["pos"], action)

        # Dùng model dự đoán trạng thái mới
        predicted_state = model(state, action)

        # Chuyển trạng thái mới thành key
        key = state_key(predicted_state)

        # Nếu trạng thái mới đã từng đi rồi
        if key in visited:

            # Lưu hướng bị lặp và step đã từng gặp
            duplicated_moves.append((action, visited[key]))

        # Nếu trạng thái mới chưa từng đi
        else:

            # Thêm hướng này vào danh sách chưa đi
            unvisited_moves.append(action)

            # Lấy tọa độ vị trí mới
            x, y = new_pos

            # Nếu ô mới là ô bẩn
            if state["floor"][x][y] == 1:

                # Thêm hướng này vào danh sách ưu tiên
                dirty_moves.append(action)

    # Nếu có hướng bị lặp
    if duplicated_moves:

        # In cảnh báo
        print("\nCẢNH BÁO!")

        # In giải thích
        print("Một số hướng sẽ quay lại trạng thái đã đi:")

        # Duyệt các hướng bị lặp
        for action, duplicated_step in duplicated_moves:

            # Nếu bị trùng với trạng thái ban đầu
            if duplicated_step == 0:

                # In thông báo trùng trạng thái ban đầu
                print(f"- Hướng {action} bị trùng với TRẠNG THÁI BAN ĐẦU")

            # Nếu bị trùng với một step nào đó
            else:

                # In thông báo trùng step cụ thể
                print(f"- Hướng {action} bị trùng với STEP {duplicated_step}")

    # Nếu có ít nhất một hướng đi tới ô bẩn chưa đi
    if len(dirty_moves) > 0:

        # Chọn random một hướng trong các hướng đi tới ô bẩn
        selected_action = random.choice(dirty_moves)

        # Trả về luật và hành động được chọn
        return {
            "condition": "Ưu tiên đi tới ô bẩn chưa đi",
            "action": selected_action
        }

    # Nếu không có ô bẩn xung quanh nhưng còn hướng chưa đi
    if len(unvisited_moves) > 0:

        # Chọn random một hướng chưa đi
        selected_action = random.choice(unvisited_moves)

        # Trả về luật và hành động được chọn
        return {
            "condition": "Còn hướng chưa đi",
            "action": selected_action
        }

    # Nếu không còn hướng nào chưa đi
    return {

        # Báo tất cả hướng đều đã đi
        "condition": "Tất cả hướng đều đã đi",

        # Trả về None để dừng
        "action": None
    }

### Model-based-reflex-agent

In [12]:
class ModelBasedReflexAgent:
    def __init__(self):
        self.state = None
        self.model = model
        self.rules = [
            "Ưu tiên đi tới ô bẩn chưa đi",
            "Còn hướng chưa đi",
            "Tất cả hướng đều đã đi",
            "Không còn bước đi hợp lệ"
        ]
        self.action = None
        self.visited = {}
        self.current_step = 0

    def program(self, percept):
        """
        MODEL-BASED-REFLEX-AGENT(percept) returns action
        """

        # state ← UPDATE-STATE(state, action, percept, model)
        self.state = update_state(
            self.state,
            self.action,
            percept,
            self.model
        )

        # lưu trạng thái hiện tại vào visited
        key = state_key(self.state)

        if key not in self.visited:
            self.visited[key] = self.current_step

        # rule ← RULE-MATCH(state, rules)
        rule = rule_match(self.state, self.rules, self.visited)

        print("\nRule được chọn:", rule["condition"])

        # action ← rule.ACTION
        self.action = rule["action"]

        # return action
        return self.action

### Main program

In [20]:
agent = ModelBasedReflexAgent()

current_state = input_state()

if current_state is None:
    print("\nDỪNG LẠI!")
    print("Lý do: Không có ô nào để máy hút bụi bắt đầu.")
else:
    print("\nTrạng thái ban đầu:")
    print_state(current_state)

    step = 0

    # Nếu vị trí ban đầu bẩn thì hút bụi ngay
    x, y = current_state["pos"]

    if current_state["floor"][x][y] == 1:
        current_state["floor"][x][y] = 0
        print(f"\nVị trí ban đầu bẩn ({x},{y}) -> Đã hút bụi")
        print("Trạng thái hiện tại sau khi hút vị trí ban đầu:")
        print_state(current_state)

    # Điều kiện dừng 1: sạch hết bụi
    if is_clean_state(current_state):
        print("\nDỪNG LẠI!")
        print("Lý do: Tất cả ô bẩn đã được hút sạch.")
    else:
        while True:
            print(f"\n========== LẦN ROLL {step + 1} ==========")

            agent.current_step = step

            percept = current_state

            action = agent.program(percept)

            # Điều kiện dừng 2: không còn hướng không lặp
            if action is None:
                print("\nDỪNG LẠI!")
                print("Lý do: Không còn hướng nào dẫn tới trạng thái chưa đi.")
                break

            print("\nAgent chọn hành động:", action)

            old_pos = current_state["pos"]
            current_state = model(current_state, action)
            new_pos = current_state["pos"]

            print(f"Máy hút bụi di chuyển từ {old_pos} sang {new_pos}.")

            step += 1

            print("\nTrạng thái sau khi thực hiện hành động:")
            print_state(current_state)

            # Điều kiện dừng 1: sạch hết bụi
            if is_clean_state(current_state):
                print("\nDỪNG LẠI!")
                print("Lý do: Tất cả ô bẩn đã được hút sạch.")
                break

    print("\nTổng số lần roll:", step)


Trạng thái ban đầu:
0 = sạch, 1 = bẩn, x = vật cản, R = máy hút bụi
['R', 1, 0, 'x']
[1, 'x', 0, 1]
[0, 'x', 1, 0]
[0, 0, 1, 0]
Vị trí máy hút bụi: (0, 0)

Vị trí ban đầu bẩn (0,0) -> Đã hút bụi
Trạng thái hiện tại sau khi hút vị trí ban đầu:
0 = sạch, 1 = bẩn, x = vật cản, R = máy hút bụi
['R', 1, 0, 'x']
[1, 'x', 0, 1]
[0, 'x', 1, 0]
[0, 0, 1, 0]
Vị trí máy hút bụi: (0, 0)

========== LẦN ROLL 1 ==========

Các bước có thể đi: ['DOWN', 'RIGHT']

Rule được chọn: Ưu tiên đi tới ô bẩn chưa đi

Agent chọn hành động: RIGHT
Máy hút bụi di chuyển từ (0, 0) sang (0, 1).

Trạng thái sau khi thực hiện hành động:
0 = sạch, 1 = bẩn, x = vật cản, R = máy hút bụi
[0, 'R', 0, 'x']
[1, 'x', 0, 1]
[0, 'x', 1, 0]
[0, 0, 1, 0]
Vị trí máy hút bụi: (0, 1)

========== LẦN ROLL 2 ==========

Các bước có thể đi: ['LEFT', 'RIGHT']

Rule được chọn: Còn hướng chưa đi

Agent chọn hành động: RIGHT
Máy hút bụi di chuyển từ (0, 1) sang (0, 2).

Trạng thái sau khi thực hiện hành động:
0 = sạch, 1 = bẩn, x = vật cả